# Calculating Power Law Distribution of Allan Factor

In [1]:
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

import plotly.express as px
import plotly.graph_objects as go

from scipy.stats import f_oneway as ftest

import warnings; warnings.filterwarnings('ignore')

In [2]:
DATA_PATH = 'allan_var.tsv'

PARAMETERS_PATH = 'av-params.csv'
# PARAMETERS_PATH = './ignore/av-params-20250825.csv'

vis_path = DATA_PATH.replace('.csv', '.html')

In [3]:
subcorpora_col = 'corpus'

In [4]:
# to rename the corpus correctly . . . 
def lang(x):
    result = str(x)
    
    if '-DISPEL-' in result:
        result = 'DISPEL'
    
    if result.startswith('GCSAusE'):
        result = 'GCSAusE'
    
    if result.startswith('se'):
        result = 'CORAAL'
        
    if result.startswith('call'):
        result = x.split('-')[0]
    
    if result.startswith('MICASE'):
        result = 'MICASE'
    
    if result.startswith('instruction'):
        result = x.split('-')[1]
    
    if result.startswith('CABNC'):
        result = 'CABNC'
        
    if result.startswith('SBCSAE'):
        result = 'SBCSAE'
        
    if result.startswith('SCoSE'):
        result = 'SCoSE'
        
    if result.startswith('candor'):
        result = 'CANDOR'
    
    if result.startswith('grace'):
        result = 'Miao-fNIRS'
    
    return result

#### Loading Results

In [5]:
results = pd.read_csv(PARAMETERS_PATH)

In [6]:
results.isna().sum()

x_turn_id            0
null                 0
self                 0
conversation_id      0
speaker              0
slope              843
b0                 751
dtype: int64

In [7]:
results.isna().mean()

x_turn_id          0.000000
null               0.000000
self               0.000000
conversation_id    0.000000
speaker            0.000000
slope              0.000741
b0                 0.000660
dtype: float64

In [8]:
(results['slope'].abs() == np.inf).sum(), (results['slope'].abs() == np.inf).mean()

(np.int64(7424), np.float64(0.00652730495456208))

In [9]:
results.head()

,x_turn_id,null,self,conversation_id,speaker,slope,b0
0,CABNC-0missing-CABNC-0missing-KB0RE004-10,False,True,CABNC-0missing-CABNC-0missing-KB0RE004,CABNC-0missing-CABNC-0missing-KB0RE004-KB0PSUN,-1.183603,-11.122748
1,CABNC-0missing-CABNC-0missing-KB0RE004-11,False,True,CABNC-0missing-CABNC-0missing-KB0RE004,CABNC-0missing-CABNC-0missing-KB0RE004-KB0PSUN,-1.979970,-6.513821
2,CABNC-0missing-CABNC-0missing-KB0RE004-12,False,True,CABNC-0missing-CABNC-0missing-KB0RE004,CABNC-0missing-CABNC-0missing-KB0RE004-KB0PSUN,-1.234872,-5.934323
3,CABNC-0missing-CABNC-0missing-KB0RE004-13,False,True,CABNC-0missing-CABNC-0missing-KB0RE004,CABNC-0missing-CABNC-0missing-KB0RE004-KB0PSUN,0.019517,-10.000541
4,CABNC-0missing-CABNC-0missing-KB0RE004-15,False,True,CABNC-0missing-CABNC-0missing-KB0RE004,CABNC-0missing-CABNC-0missing-KB0RE004-KB0PSUN,-0.995607,-9.275234


In [10]:
sel = results['slope'].isna() | (results['slope'].abs() == np.inf) 
sel.sum(), sel.mean(), (~sel).sum()

(np.int64(8267), np.float64(0.007268484652392876), np.int64(1129109))

In [11]:
results['self'].value_counts()

self
True     572655
False    564721
Name: count, dtype: int64

In [12]:
results['x_turn_id'].loc[sel].nunique()

8153

In [13]:
# is_null = np.array(['null-' in x for x in tqdm(results['x_turn_id'])])
is_null = results['null']
is_null.sum()

np.int64(99856)

Adding Corpus names back in . . .

In [14]:
results[subcorpora_col] = [lang(x) for x in tqdm(results['conversation_id'])]

  0%|          | 0/1137376 [00:00<?, ?it/s]

In [15]:
results[subcorpora_col].value_counts()

corpus
CABNC          331432
CORAAL         298555
MICASE         241735
callhome        79159
SBCSAE          69885
callfriend      36373
Miao-fNIRS      34137
CANDOR          17709
SCoSE           15521
GCSAusE          7134
DISPEL           4380
Graesser          871
med_school        361
Frederiksen       124
Name: count, dtype: int64

In [16]:
results.head()

,x_turn_id,null,self,conversation_id,speaker,slope,b0,corpus
0,CABNC-0missing-CABNC-0missing-KB0RE004-10,False,True,CABNC-0missing-CABNC-0missing-KB0RE004,CABNC-0missing-CABNC-0missing-KB0RE004-KB0PSUN,-1.183603,-11.122748,CABNC
1,CABNC-0missing-CABNC-0missing-KB0RE004-11,False,True,CABNC-0missing-CABNC-0missing-KB0RE004,CABNC-0missing-CABNC-0missing-KB0RE004-KB0PSUN,-1.979970,-6.513821,CABNC
2,CABNC-0missing-CABNC-0missing-KB0RE004-12,False,True,CABNC-0missing-CABNC-0missing-KB0RE004,CABNC-0missing-CABNC-0missing-KB0RE004-KB0PSUN,-1.234872,-5.934323,CABNC
3,CABNC-0missing-CABNC-0missing-KB0RE004-13,False,True,CABNC-0missing-CABNC-0missing-KB0RE004,CABNC-0missing-CABNC-0missing-KB0RE004-KB0PSUN,0.019517,-10.000541,CABNC
4,CABNC-0missing-CABNC-0missing-KB0RE004-15,False,True,CABNC-0missing-CABNC-0missing-KB0RE004,CABNC-0missing-CABNC-0missing-KB0RE004-KB0PSUN,-0.995607,-9.275234,CABNC


## Aggregate corpus statistics

#### Self values

What is the aggregate slope for the whole corpus with regards to self? Let's find that out here! The following tests what the slope is for "self" in the data for both the real data and the null-distribution data separately. Both are reported on in the final paper story.

##### Real data

In [17]:
sel_ = ~sel & (results['self']) & (~is_null)
sel_.sum()

np.int64(519390)

In [18]:
mu, SE = (results['slope'].loc[sel_].mean(), results['slope'].loc[sel_].std()/np.sqrt(sel_.sum()))
mu, SE

(np.float64(-0.19898927398635263), np.float64(0.0013793206576512773))

t-test for good measure.

In [19]:
mu / SE

np.float64(-144.26614499140018)

##### Null data

In [20]:
sel_ = (~sel) & (results['self']) & (is_null)

mu, SE = (results['slope'].loc[sel_].mean(), results['slope'].loc[sel_].std()/np.sqrt(sel_.sum()))

T = mu / SE

In [21]:
sel_.sum(), mu, SE, T

(np.int64(49862),
 np.float64(-0.026184766234679663),
 np.float64(0.004122426379758867),
 np.float64(-6.351785046604347))

#### Other values

What is the aggregate slope for the whole corpus with regards to others? Let's find that out here! The following tests what the slope is for "other" in the data for both the real data and the null-distribution data separately. Both are reported on in the final paper story.

##### Real data

In [22]:
sel_ = ~sel & (~results['self']) & (~is_null)
sel_.sum()

np.int64(509863)

In [23]:
mu, SE = (results['slope'].loc[sel_].mean(), results['slope'].loc[sel_].std()/np.sqrt(sel_.sum()))
mu, SE

(np.float64(-0.11435293747127544), np.float64(0.001441578110797366))

t-test for good measure.

In [24]:
mu / SE

np.float64(-79.3248292373311)

##### Null data

In [25]:
sel_ = ~sel & (~results['self']) & (is_null)

mu, SE = (results['slope'].loc[sel_].mean(), results['slope'].loc[sel_].std()/np.sqrt(sel_.sum()))

T = mu / SE

In [26]:
sel_.sum(), mu, SE, T

(np.int64(49994),
 np.float64(-0.014104193641448248),
 np.float64(0.0039876401499135605),
 np.float64(-3.53697753839548))

## Removing null data for other statistics

In [27]:
print(results.shape)
results = results.loc[~is_null]
results.shape

(1137376, 8)


(1037520, 8)

## individual corpora statistics

In [28]:
subcorpus_results = []

In [29]:
for corpus in tqdm(results[subcorpora_col].unique()):
    
    res = {'corpus': corpus}
    
    sub = results.loc[
        results[subcorpora_col].isin([corpus])
        & (~sel) & (~is_null)
    ]
    
    # self vals
    sel__ = sub['self']
    mu, se, n = sub['slope'].loc[sel__].mean(), sub['slope'].loc[sel__].std() / np.sqrt(sel__.sum()), sel__.sum()
    res['self_alpha'] = mu
    res['self_se'] = se
    res['self_n'] = n
    res['self_t'] = mu/se
    
    # self vals
    sel__ = ~sub['self']
    mu, se, n = sub['slope'].loc[sel__].mean(), sub['slope'].loc[sel__].std() / np.sqrt(sel__.sum()), sel__.sum()
    res['other_alpha'] = mu
    res['other_se'] = se
    res['other_n'] = n
    res['other_t'] = mu/se
    
    # save outputs
    subcorpus_results += [res]

subcorpus_results = pd.DataFrame(subcorpus_results)    

  0%|          | 0/14 [00:00<?, ?it/s]

In [30]:
subcorpus_results.head(n=100)

,corpus,self_alpha,self_se,self_n,self_t,other_alpha,other_se,other_n,other_t
0,CABNC,-0.173427,0.002956,147795,-58.667278,-0.125327,0.002771,152696,-45.227012
1,GCSAusE,-0.241742,0.022936,3149,-10.540027,-0.137447,0.019384,3310,-7.090647
2,MICASE,-0.184033,0.002705,112594,-68.038848,-0.101279,0.003025,107387,-33.475832
3,SBCSAE,-0.213517,0.005895,31411,-36.221569,-0.109391,0.005924,31177,-18.465531
4,SCoSE,-0.247887,0.011993,6986,-20.669973,-0.065708,0.013489,5703,-4.871082
5,callfriend,-0.179798,0.007871,16438,-22.842324,-0.089555,0.008431,16299,-10.622422
6,callhome,-0.205203,0.005244,36872,-39.130649,-0.086497,0.005692,33779,-15.196606
7,CANDOR,-0.124898,0.011719,8646,-10.657960,-0.066118,0.011641,8720,-5.679955
8,Miao-fNIRS,-0.158055,0.007919,15429,-19.960113,-0.121710,0.007829,15408,-15.546170
9,DISPEL,-0.102802,0.025926,2012,-3.965216,-0.074424,0.028198,1954,-2.639354


In [31]:
subcorpus_results = []

In [32]:
for corpus in tqdm(results[subcorpora_col].unique()):
    
    sub = results.loc[
        results[subcorpora_col].isin([corpus])
        & ~sel
    ]
    
    # self vals
    res = {'corpus': corpus}
    
    sel__ = sub['self']
    mu, se, n = sub['slope'].loc[sel__].mean(), sub['slope'].loc[sel__].std() / np.sqrt(sel__.sum()), sel__.sum()
    res['self'] = True
    res['alpha'] = mu
    res['se'] = se
    res['n'] = n
    res['t'] = mu/se
    
    subcorpus_results += [res]
    
    # self vals
    res = {'corpus': corpus}
    
    sel__ = ~sub['self']
    mu, se, n = sub['slope'].loc[sel__].mean(), sub['slope'].loc[sel__].std() / np.sqrt(sel__.sum()), sel__.sum()
    res['self'] = False
    res['alpha'] = mu
    res['se'] = se
    res['n'] = n
    res['t'] = mu/se
    
    # save outputs
    subcorpus_results += [res]

subcorpus_results = pd.DataFrame(subcorpus_results)    

  0%|          | 0/14 [00:00<?, ?it/s]

In [33]:
subcorpus_results.head()

,corpus,self,alpha,se,n,t
0,CABNC,True,-0.173427,0.002956,147795,-58.667278
1,CABNC,False,-0.125327,0.002771,152696,-45.227012
2,GCSAusE,True,-0.241742,0.022936,3149,-10.540027
3,GCSAusE,False,-0.137447,0.019384,3310,-7.090647
4,MICASE,True,-0.184033,0.002705,112594,-68.038848


In [34]:
f = open('table.txt', 'w')

self_vals = subcorpus_results[[col for col in list(subcorpus_results) if col not in ['self']]].loc[subcorpus_results['self']].to_latex(index=False)
self_vals = self_vals.replace('\\toprule', '\\hline').replace('\\midrule', '\\hline\\hline').replace('\\bottomrule', '\\hline')
self_vals = self_vals.replace('\\\\', '\\\\\hline').replace('|', '$|$').replace('alpha', ' $\\alpha$')

other_vals = subcorpus_results[[col for col in list(subcorpus_results) if col not in ['self']]].loc[subcorpus_results['self']].to_latex(index=False, header=False)
other_vals = other_vals.replace('\\toprule', '\\hline').replace('\\midrule', '\\hline\\hline').replace('\\bottomrule', '\\hline')
other_vals = other_vals.replace('\\\\', '\\\\\hline').replace('|', '$|$').replace('alpha', ' $\\alpha$')

f.write(self_vals + '\n\\\multicolumn{5}{|r|}{\\textit{Other}}\\\\ \n' + other_vals)
f.close()


In [ ]:
# reporting['Var'] = reporting.index.values
# 
# with open(lme_results_name.replace('.csv', '2.txt'), 'w') as f:
#     txt =  reporting[['Var', 'coefs', 'stat', 'p']].to_latex(index=False).replace('\\toprule', '\\hline').replace('\\midrule', '\\hline\\hline').replace('\\bottomrule', '\\hline')
# 
#     txt = txt.replace('\\\\', '\\\\\hline').replace('|', '$|$').replace('_delta', ' $\Delta$')
#     f.write(txt)
#     f.close()

## Individual differences between speakers

### Self
In effect, we are testing whether or not there are individual differences in the degree to which speakers drift from their own ideas from turn to turn.

To test this, we first calculate the mean difference of the exponential rate of decay ($\alpha$) for all speakers compared to all other speakers. We then estimate the variance in rate of decay as a direct measurement of individual differences.

In [ ]:
sel_ = ~sel & (results['self']) #& (~is_null)
sel_.sum()

In [ ]:
df = results[['speaker', 'slope']].loc[sel_].groupby('speaker').agg('mean').reset_index()

In [ ]:
df.head()

In [ ]:
delta_xy = np.abs(df['slope'].values - df['slope'].values.reshape(-1,1))
delta_xy = np.tril(delta_xy)
N = (delta_xy != 0)
N.sum()

In [ ]:
mu = delta_xy.sum() / N.sum()
mu

In [ ]:
var = (((delta_xy - mu) * N )**2).sum()/(N.sum() - 1)
np.sqrt(var)

In [ ]:
SE = np.sqrt(var/(N.sum()-1))
SE

In [ ]:
mu / np.sqrt(var/N.sum())

#### Much more basic T-Test

In [ ]:
mu = df['slope'].mean()
se = df['slope'].std() / np.sqrt(df.shape[0])
mu, se, df.shape[0], mu/se

#### ANOVA test

In [ ]:
# means
df = results[['speaker', 'slope']].loc[sel_].groupby(by=['speaker']).agg('mean').reset_index()

# n
df['n'] = results[['speaker', 'slope']].loc[sel_].groupby(by=['speaker']).agg('count').reset_index()['slope']

# SSW
df['ssw'] = results[['speaker', 'slope']].loc[sel_].groupby(by=['speaker']).agg(lambda x: ((x-x.mean())**2).sum()).reset_index()['slope']

# SSB
df['ssb'] = (df['slope'] - df['slope'].mean())**2

In [ ]:
df.head()

In [ ]:
dof_b, dof_w = df.shape[0]-1, sel_.sum() - df.shape[0]
dof_b, dof_w

In [ ]:
F = (df['ssb'].sum() / dof_b) / (df['ssw'].sum() / dof_w)
F

### Other

Are there individual differences in the degree to which individual speakers influence their conversation partners? We replicate the same procedure in the "self" approach above on data for exponential decay rate ($\alpha$) for the influence of an utterance on the "other".

In [ ]:
sel_ = ~sel & (~results['self']) #& (~is_null)
sel_.sum()

In [ ]:
df = results[['speaker', 'slope']].loc[sel_].groupby('speaker').agg('mean').reset_index()

In [ ]:
df.head()

In [ ]:
delta_xy = np.abs(df['slope'].values - df['slope'].values.reshape(-1,1))
delta_xy = np.tril(delta_xy)
N = (delta_xy != 0)
N.sum()

In [ ]:
mu = delta_xy.sum() / N.sum()
mu

In [ ]:
var = (((delta_xy - mu) * N) ** 2).sum() / (N.sum() - 1)
np.sqrt(var)

In [ ]:
SE = np.sqrt(var/(N.sum()-1))
SE

In [ ]:
mu / np.sqrt(var/N.sum())